In [1]:
import math
import random
from concurrent.futures import ThreadPoolExecutor

# Global parameters for PCA
GRID_ROWS, GRID_COLS = 4, 4  # 4x4 Cellular Grid (16 cells)
MAX_ITERATIONS = 30
ENERGY_PER_UNIT = 1.2        # Battery consumed per unit distance

# Helper: Calculate Euclidean distance between two points
def distance(locations, a, b):
    x1, y1 = locations[a]
    x2, y2 = locations[b]
    return math.hypot(x2 - x1, y2 - y1)

# Function 1: Evaluate total distance, battery usage, and fitness
def calculate_fitness(locations, route, max_battery):
    full_path = [0] + route + [0]
    total_dist = sum(distance(locations, full_path[i], full_path[i+1]) for i in range(len(full_path) - 1))
    battery_used = total_dist * ENERGY_PER_UNIT

    # Add heavy penalty if battery limit is exceeded
    penalty = 1000.0 if battery_used > max_battery else 0.0
    return total_dist + penalty, total_dist, battery_used

# Function 2: Find best neighbor (von Neumann: up, down, left, right)
def get_best_neighbor(grid, r, c):
    neighbors = []
    for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
        nr, nc = (r + dr) % GRID_ROWS, (c + dc) % GRID_COLS
        neighbors.append(grid[nr][nc])
    return min(neighbors, key=lambda cell: cell["fitness"])

# Function 3: Update route toward best neighbor
def update_route(current_route, best_neighbor_route):
    size = len(current_route)
    if size < 2:
        return current_route[:]

    start, end = sorted(random.sample(range(size), 2))

    # Copy a segment from the best neighbor
    child = [None] * size
    child[start:end] = best_neighbor_route[start:end]

    # Fill remaining regions in original order
    remaining = [gene for gene in current_route if gene not in child]
    idx = 0
    for i in range(size):
        if child[i] is None:
            child[i] = remaining[idx]
            idx += 1

    # Small random swap (20% chance) to explore shorter paths
    if random.random() < 0.2:
        i, j = random.sample(range(size), 2)
        child[i], child[j] = child[j], child[i]

    return child

# Function 4: Take dynamic user inputs
def get_user_inputs():
    print("=== Agricultural Drone Path Optimization (User Input) ===")
    locations = {}

    cx, cy = map(float, input("Enter Charging Station coordinates (x y): ").split())
    locations[0] = (cx, cy)

    num_regions = int(input("Enter number of farm regions to survey: "))
    for i in range(1, num_regions + 1):
        rx, ry = map(float, input(f"Enter coordinates for Region {i} (x y): ").split())
        locations[i] = (rx, ry)

    max_battery = float(input("Enter drone maximum battery capacity (units): "))
    return locations, num_regions, max_battery

# Main Parallel Cellular Algorithm
def parallel_cellular_drone_routing():
    locations, num_regions, max_battery = get_user_inputs()
    regions = list(range(1, num_regions + 1))

    # Initialize grid with random routes
    grid = []
    for _ in range(GRID_ROWS):
        row = []
        for _ in range(GRID_COLS):
            route = random.sample(regions, len(regions))
            fit, dist, batt = calculate_fitness(locations, route, max_battery)
            row.append({"route": route, "fitness": fit, "dist": dist, "batt": batt})
        grid.append(row)

    # Function to update a single cell (executed in parallel)
    def process_cell(pos):
        r, c = pos
        best_neighbor = get_best_neighbor(grid, r, c)
        new_route = update_route(grid[r][c]["route"], best_neighbor["route"])
        fit, dist, batt = calculate_fitness(locations, new_route, max_battery)

        if fit < grid[r][c]["fitness"]:
            return (r, c, {"route": new_route, "fitness": fit, "dist": dist, "batt": batt})
        return (r, c, grid[r][c])

    positions = [(r, c) for r in range(GRID_ROWS) for c in range(GRID_COLS)]

    # Run optimization loop in parallel
    with ThreadPoolExecutor() as executor:
        for _ in range(MAX_ITERATIONS):
            updates = list(executor.map(process_cell, positions))
            for r, c, updated_cell in updates:
                grid[r][c] = updated_cell

    # Extract global best cell from the grid
    all_cells = [cell for row in grid for cell in row]
    best_cell = min(all_cells, key=lambda cell: cell["fitness"])

    optimal_path = [0] + best_cell["route"] + [0]
    print("\n=== Optimization Results ===")
    print(f"Optimal Visiting Sequence : {' -> '.join(map(str, optimal_path))}")
    print(f"Total Flight Distance     : {best_cell['dist']:.2f} units")
    print(f"Battery Consumed          : {best_cell['batt']:.2f} / {max_battery} units")
    print(f"Battery Constraint Status : {'SAFE' if best_cell['batt'] <= max_battery else 'EXCEEDED'}")

if __name__ == "__main__":
    parallel_cellular_drone_routing()


=== Agricultural Drone Path Optimization (User Input) ===
Enter Charging Station coordinates (x y): 0 0 
Enter number of farm regions to survey: 5
Enter coordinates for Region 1 (x y): 2 4
Enter coordinates for Region 2 (x y): 5 2
Enter coordinates for Region 3 (x y): 6 6 
Enter coordinates for Region 4 (x y): 1 7
Enter coordinates for Region 5 (x y): 4 5
Enter drone maximum battery capacity (units): 30

=== Optimization Results ===
Optimal Visiting Sequence : 0 -> 1 -> 4 -> 5 -> 3 -> 2 -> 0
Total Flight Distance     : 22.98 units
Battery Consumed          : 27.58 / 30.0 units
Battery Constraint Status : SAFE
